# Task 2 - Lego Minifigs
In this part of the notebook, we start the multiclass image classification pipeline for the Lego Minifigs. The goal of this task is to predict the category of a minifig based on its image. To do this in a structured way, we first prepare the environment for data loading, exploration, preprocessing, and dataset splitting. This fits the standard supervised learning workflow from the course: first understand the data, then prepare the input and target correctly, and only after that move on to model building and evaluation.


In [29]:
import sys
print(sys.executable)

/usr/bin/python3


# Part 1.1 - training our own CNN
The first code block imports the libraries needed for the early stages of the pipeline. These packages are used to read the metadata, handle images, organise the dataset in tabular form, create train, validation, and test splits, and visualise distributions in the data. This is an important setup step, because a correct deep learning workflow starts with careful data preparation and inspection before training a model.

In [30]:
import os
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

### Reproducibility

A fixed random seed is set at the start so that later steps remain consistent across runs.

### Loading the metadata

In this step, we load the Lego Minifigs metadata from the provided JSON file and convert it into a pandas DataFrame. The assignment states that the JSON file contains one entry per image, so this is the logical starting point of the pipeline.

### First inspection

Next, we inspect the structure of the dataset by printing the number of rows, the number of columns, the column names, and the first records. This gives a quick overview of the available instances and variables before moving on to preprocessing and modeling. In the course terminology, the rows represent instances and the columns represent variables that can later be used as predictors or targets.


In [31]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

# JSON_PATH = "data/minifigs.json"
# IMAGE_ROOT = "data/images"

from google.colab import drive
drive.mount('/content/drive')
JSON_PATH = "/content/drive/MyDrive/Colab_Notebooks/Project/data/minifigs.json"
IMAGE_ROOT = "/content/drive/MyDrive/Colab_Notebooks/Project/data/images"
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
df.head()

print(df.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of rows: 17362
Number of columns: 15
['id', 'name', 'link', 'year', 'img_url', 'minifig_number', 'category', 'subcategory', 'year_released', 'set_id', 'current_value_new', 'current_value_used', 'character_name', 'img_local_path', 'themes']


### Exploring the target distribution and checking missing values

In this step, we further inspect the structure of the classification task by looking at the number of unique categories, the most frequent categories, and the number of unique subcategories. This gives a clearer view of the target distribution before training a model. We also check for missing values in the most relevant columns, since detecting data quality issues is an important preprocessing step before building the final pipeline.

In [32]:
print("Number of unique categories:", df["category"].nunique())
print()
print("Top 20 categories:")
print(df["category"].value_counts().head(20))

print("Missing values per relevant column:")
print(df[["category", "img_local_path"]].isna().sum())
print("Number of unique subcategories:", df["subcategory"].nunique())

Number of unique categories: 122

Top 20 categories:
category
Town                          3522
Star Wars                     1496
Super Heroes                  1110
DUPLO                          987
NINJAGO                        944
Friends                        887
Collectible Minifigures        804
Harry Potter                   636
Castle                         543
Holiday & Event                383
Disney                         375
LEGO Ideas                     272
BrickLink Designer Program     264
Minecraft                      253
Sports                         252
Super Mario                    229
Space                          209
Pirates                        198
Monkie Kid                     180
LEGO Brand                     162
Name: count, dtype: int64
Missing values per relevant column:
category          0
img_local_path    1
dtype: int64
Number of unique subcategories: 518


### Removing incomplete observations

In this step, we remove observations with missing values in `category` or `img_local_path`. These columns are essential for the multiclass task, because the model needs both a valid target label and a valid reference to the corresponding image. Handling missing values is an important preprocessing step before continuing with the pipeline, since incomplete observations cannot be used reliably for training or evaluation.

In [33]:
df = df.dropna(subset=["category", "img_local_path"]).copy()

print("Number of rows after dropping missing category/img_local_path:", len(df))

Number of rows after dropping missing category/img_local_path: 17361


### Constructing full image paths and verifying file availability

In this step, we convert the relative image paths from the metadata into full local file paths. This is necessary because the dataset information and the actual image files are stored separately, so we need a correct path that allows the notebook to locate each image on disk.

After creating these full paths, we perform a file existence check. This makes it possible to verify whether the images referenced in the metadata are actually available in the local image folder. This is an important preprocessing step, because missing image files would cause problems later when building the dataset and training the model.

In [34]:
def build_full_path(img_local_path, image_root=IMAGE_ROOT):
    filename = os.path.basename(img_local_path)
    return os.path.join(image_root, filename)

df["full_path"] = df["img_local_path"].apply(build_full_path)

df[["img_local_path", "full_path"]].head()

df["file_exists"] = df["full_path"].apply(os.path.exists)

print(df["file_exists"].value_counts(dropna=False))

file_exists
True    17361
Name: count, dtype: int64


### Inspecting the category distribution

In this step, we analyse how the observations are distributed across the different categories. We first count how many categories are present, then summarise how many examples are available per category, and finally inspect both the smallest and largest categories.

This helps us understand whether the multiclass problem is balanced or whether some categories have only a limited number of examples. That is an important check before training, because very small classes may be difficult for the model to learn reliably and can affect the quality of the final results.

In [35]:
category_counts = df["category"].value_counts()

print("Aantal categorieën:", len(category_counts))
print()
print("Beschrijvende statistiek van aantal voorbeelden per categorie:")
print(category_counts.describe())

thresholds = [5, 10, 20, 50, 100]

for t in thresholds:
    n_classes = (category_counts < t).sum()
    print(f"Aantal categorieën met minder dan {t} voorbeelden: {n_classes}")



print("10 kleinste categorieën:")
print(category_counts.sort_values().head(10))

print("\n10 grootste categorieën:")
print(category_counts.head(10))

Aantal categorieën: 122

Beschrijvende statistiek van aantal voorbeelden per categorie:
count     122.000000
mean      142.303279
std       390.177410
min         1.000000
25%        13.250000
50%        31.000000
75%        85.000000
max      3522.000000
Name: count, dtype: float64
Aantal categorieën met minder dan 5 voorbeelden: 11
Aantal categorieën met minder dan 10 voorbeelden: 24
Aantal categorieën met minder dan 20 voorbeelden: 40
Aantal categorieën met minder dan 50 voorbeelden: 78
Aantal categorieën met minder dan 100 voorbeelden: 94
10 kleinste categorieën:
category
Discovery              1
Fusion                 1
Quatro                 1
Universe               1
Nike                   2
Back to the Future     2
Architecture           2
Homemaker              2
Clikits                3
The Legend of Zelda    4
Name: count, dtype: int64

10 grootste categorieën:
category
Town                       3522
Star Wars                  1496
Super Heroes               1110
DUPLO     

### Filtering rare categories

In this step, we keep only categories with at least a minimum number of examples. Categories with fewer than 20 samples are removed from the dataset, and we then check how many categories and observations remain after filtering.

This filtering step makes the multiclass task more stable, because classes with only a very small number of examples are difficult to learn reliably. By keeping only categories with enough data, we create a cleaner classification problem and reduce the effect of extremely sparse classes on model training and evaluation.

In [36]:
MIN_SAMPLES_PER_CLASS = 20

category_counts = df["category"].value_counts()
valid_classes = category_counts[category_counts >= MIN_SAMPLES_PER_CLASS].index

df_filtered = df[df["category"].isin(valid_classes)].copy()

print("Number of categories before filtering:", df["category"].nunique())
print("Number of categories after filtering:", df_filtered["category"].nunique())
print("Number of rows after filtering:", len(df_filtered))

kept_fraction = len(df_filtered) / len(df)

print(f"Fraction of kept data: {kept_fraction:.4f}")
print(f"Percentage of kept data: {kept_fraction * 100:.2f}%")

Number of categories before filtering: 122
Number of categories after filtering: 82
Number of rows after filtering: 17028
Fraction of kept data: 0.9808
Percentage of kept data: 98.08%


### Encoding the target labels

In this step, we convert the category names into numeric class labels. We first create a sorted list of the remaining categories, then build mappings from category name to index and from index back to category name. After that, we add the numeric label to the dataset and keep only the columns that are needed for modeling.

This step is necessary because the model cannot work directly with text labels. The classification target must be represented numerically so that each image can be linked to a class index during training and evaluation. We also inspect the filtered class distribution once more to confirm that the dataset is ready for the next stage of the pipeline.

In [37]:
classes = sorted(df_filtered["category"].unique())
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
idx_to_class = {idx: cls_name for cls_name, idx in class_to_idx.items()}

df_filtered["label"] = df_filtered["category"].map(class_to_idx)

print("Aantal klassen na encoding:", len(classes))
print("Voorbeeld mapping:", list(class_to_idx.items())[:10])

model_df = df_filtered[["id", "name", "category", "label", "full_path"]].copy()

print(model_df.shape)
model_df.head()

filtered_counts = df_filtered["category"].value_counts()

print("10 kleinste categorieën na filtering:")
print(filtered_counts.sort_values().head(10))

Aantal klassen na encoding: 82
Voorbeeld mapping: [('Adventurers', 0), ('Agents', 1), ('Alpha Team', 2), ('Animal Crossing', 3), ('Aquazone', 4), ('Atlantis', 5), ('Avatar', 6), ('BIONICLE', 7), ('Batman I', 8), ('Belville', 9)]
(17028, 5)
10 kleinste categorieën na filtering:
category
The Angry Birds Movie    20
Hero Factory             20
Ninja                    20
The Lone Ranger          21
Monster Fighters         22
Western                  23
Trolls World Tour        23
Exo-Force                24
Stranger Things          24
One Piece                25
Name: count, dtype: int64


### Creating the train, validation, and test split

In this step, we split the dataset into training, validation, and test sets. We first separate a test set, and then split the remaining data again to create a validation set, so that the final proportions correspond to the intended setup.

We use stratified splitting based on the class labels. This means that the class distribution is preserved as much as possible across the different subsets. This is important in a multiclass setting, because it ensures that training, validation, and test data remain comparable and that evaluation is based on a representative distribution of categories.

In [38]:
train_val_df, test_df = train_test_split(
    model_df,
    test_size=0.15,
    stratify=model_df["label"],
    random_state=42
)

print("train_val_df shape:", train_val_df.shape)
print("test_df shape:", test_df.shape)

val_relative_size = 0.15 / 0.85  # zodat validation uiteindelijk 15% van totaal is

train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_relative_size,
    stratify=train_val_df["label"],
    random_state=42
)

print("train_df shape:", train_df.shape)
print("val_df shape:", val_df.shape)
print("test_df shape:", test_df.shape)

train_val_df shape: (14473, 5)
test_df shape: (2555, 5)
train_df shape: (11918, 5)
val_df shape: (2555, 5)
test_df shape: (2555, 5)


### Verifying the data split

After creating the train, validation, and test sets, we check their sizes and relative proportions with respect to the full dataset. We also verify how many unique classes are present in each subset.

This is a useful consistency check before moving on to model training. It allows us to confirm that the split was created as intended and that all subsets still contain the full set of classes needed for a valid multiclass classification setup.

In [39]:
total = len(model_df)

print(f"Train: {len(train_df)} ({len(train_df)/total:.4f})")
print(f"Val:   {len(val_df)} ({len(val_df)/total:.4f})")
print(f"Test:  {len(test_df)} ({len(test_df)/total:.4f})")

print("Number of unique classes in train:", train_df["label"].nunique())
print("Number of unique classes in val:", val_df["label"].nunique())
print("Number of unique classes in test:", test_df["label"].nunique())

Train: 11918 (0.6999)
Val:   2555 (0.1500)
Test:  2555 (0.1500)
Number of unique classes in train: 82
Number of unique classes in val: 82
Number of unique classes in test: 82


### Comparing class distributions across the splits

In this step, we compare the relative class distributions of the training, validation, and test sets. We calculate the proportion of each category in every subset, combine these distributions into a single table, and then measure the average absolute difference between them.

This allows us to check whether the stratified split worked as intended. If the differences between the subsets remain small, the train, validation, and test sets can be considered well aligned, which makes later model evaluation more reliable. Finally, we save the three splits to disk so they can be reused consistently in the rest of the workflow.

In [40]:
def class_distribution(df_part, label_col="category"):
    return df_part[label_col].value_counts(normalize=True).sort_index()

train_dist = class_distribution(train_df)
val_dist = class_distribution(val_df)
test_dist = class_distribution(test_df)

dist_compare = pd.DataFrame({
    "train": train_dist,
    "val": val_dist,
    "test": test_dist
}).fillna(0)

dist_compare.head(15)

print("Mean absolute difference train-val:",
      np.mean(np.abs(dist_compare["train"] - dist_compare["val"])))

print("Mean absolute difference train-test:",
      np.mean(np.abs(dist_compare["train"] - dist_compare["test"])))

os.makedirs("outputs_multiclass", exist_ok=True)

train_df.to_csv("outputs_multiclass/train_split.csv", index=False)
val_df.to_csv("outputs_multiclass/val_split.csv", index=False)
test_df.to_csv("outputs_multiclass/test_split.csv", index=False)

print("Splits saved in outputs_multiclass/")

Mean absolute difference train-val: 0.00012913004587879217
Mean absolute difference train-test: 0.00012043380230278475
Splits saved in outputs_multiclass/


### Building the image pipeline

In this step, we prepare the PyTorch input pipeline for multiclass image classification. We first define the device that will be used for computation, so the model can run on either the GPU or the CPU depending on availability.

Next, we define separate image transforms for training and evaluation. The training pipeline includes resizing, random horizontal flipping, small rotations, and color jittering, which introduce controlled variation in the input images. This helps the model become less sensitive to small visual changes and improves robustness during training. For validation and test data, we only apply resizing and normalization, since these sets should reflect the original data as consistently as possible.

We then create a custom dataset class that loads each image from its file path and returns both the processed image and its numeric class label. After that, we construct separate datasets and dataloaders for the training, validation, and test sets. The dataloaders organise the data into batches, which makes training and evaluation more efficient.

Finally, we perform a few sanity checks by inspecting the dataset sizes, the shape of one batch, and one individual sample. This confirms that the input pipeline is working correctly before the model is defined and trained.

In [41]:
import torch

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Image transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Custom Dataset
class MinifigDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["full_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label

# Make datasets
train_dataset = MinifigDataset(train_df, transform=train_transforms)
val_dataset = MinifigDataset(val_df, transform=eval_transforms)
test_dataset = MinifigDataset(test_df, transform=eval_transforms)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

# Set up DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# Sanity check on 1 batch
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First 10 labels:", labels[:10].tolist())

# number of classes
num_classes = len(classes)
print("Number of classes:", num_classes)

# Optional extra sanity check on one sample
sample_image, sample_label = train_dataset[0]
print("Single image tensor shape:", sample_image.shape)
print("Single label:", sample_label)

Device: cuda
Train dataset size: 11918
Val dataset size: 2555
Test dataset size: 2555
Image batch shape: torch.Size([64, 3, 224, 224])
Label batch shape: torch.Size([64])
First 10 labels: [30, 61, 26, 58, 26, 38, 13, 3, 71, 58]
Number of classes: 82
Single image tensor shape: torch.Size([3, 224, 224])
Single label: 26


### Defining the baseline CNN and the training procedure

In this step, we define the first model that will serve as our baseline for the multiclass classification task. The architecture is a simple convolutional neural network with three convolutional blocks for feature extraction, followed by a fully connected classifier that maps the extracted image features to the final category predictions. This gives us a clear reference model before moving to more advanced architectures later in the notebook.

After defining the model, we specify the loss function and the optimizer. Because this is a multiclass classification problem with one target label per image, the model is trained with a standard classification loss. We then define separate functions for one training epoch and for evaluation, so that model performance can be tracked in a consistent way across the training and validation sets.

The training loop stores the most important metrics for each epoch and uses early stopping based on the validation macro-F1 score. This means the best model is selected according to validation performance, while training is stopped when no further improvement is observed. This helps limit unnecessary training and reduces the risk of overfitting.

Finally, we run a small sanity check on one batch to verify that the output shape of the model matches the expected number of classes. This confirms that the architecture is correctly connected before the actual training starts.

In [42]:
from sklearn.metrics import accuracy_score, f1_score
import torch.nn.functional as F

# Baseline CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 224 -> 112

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 112 -> 56

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)    # 56 -> 28
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Model, loss and optimizer
baseline_model = SimpleCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print(baseline_model)

# Train function for 1 epoch
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1

# Evaluation function
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1, all_labels, all_preds

# Training loop with early stopping on validation macro-F1
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=20,
    patience=5,
    model_path="outputs_multiclass/baseline_cnn_best.pt"
):
    best_f1 = -1.0
    patience_counter = 0
    history = []

    for epoch in range(epochs):
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc, val_f1, _, _ = evaluate(
            model, val_loader, criterion, device
        )

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1
        }
        history.append(row)

        print(
            f"Epoch {epoch+1:02d} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | train_f1={train_f1:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    history_df = pd.DataFrame(history)
    return history_df

# Short sanity check: forward pass op one batch
baseline_model.eval()
with torch.no_grad():
    sample_outputs = baseline_model(images.to(device))

print("Output shape:", sample_outputs.shape)
print("Expected shape:", (images.shape[0], num_classes))

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=256, out_features=82, bias=True)
  )
)
Output shape: torch.Size([64, 82])
Expected shape: (64, 82)


### Training the baseline model and tracking its performance

In this step, we train the baseline CNN on the training set and evaluate it after each epoch on the validation set. The full training history is stored so that we can later inspect how the model evolved over time and identify the epoch with the best validation macro-F1 score.

We also save the training history to disk, which makes the experiment easier to document and reproduce. After training, we select the best epoch based on validation macro-F1, since this is the metric used for model selection in the training procedure.

Finally, we visualise the evolution of loss, accuracy, and macro-F1 for both the training and validation sets. These plots help us assess whether the model is learning effectively and whether there are signs of underfitting or overfitting during training.

In [ ]:
# Train the baseline
baseline_history = train_model(
    model=baseline_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=20,
    patience=5,
    model_path="outputs_multiclass/baseline_cnn_best.pt"
)

# Look at history
print("\nTrainingshistoriek:")
print(baseline_history)

# Save history to CSV
baseline_history.to_csv("outputs_multiclass/baseline_history.csv", index=False)
print("\nHistory saved as outputs_multiclass/baseline_history.csv")

# Beste epoch bepalen op basis van validation macro-F1
best_idx = baseline_history["val_f1"].idxmax()
best_row = baseline_history.loc[best_idx]

print("\nBest epoch given validation macro-F1:")
print(best_row)

# Plot 1: loss
plt.figure(figsize=(8, 5))
plt.plot(baseline_history["epoch"], baseline_history["train_loss"], label="Train loss")
plt.plot(baseline_history["epoch"], baseline_history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline CNN - Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: accuracy
plt.figure(figsize=(8, 5))
plt.plot(baseline_history["epoch"], baseline_history["train_acc"], label="Train accuracy")
plt.plot(baseline_history["epoch"], baseline_history["val_acc"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline CNN - Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# Plot 3: macro-F1
plt.figure(figsize=(8, 5))
plt.plot(baseline_history["epoch"], baseline_history["train_f1"], label="Train macro-F1")
plt.plot(baseline_history["epoch"], baseline_history["val_f1"], label="Validation macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Baseline CNN - Macro-F1")
plt.legend()
plt.tight_layout()
plt.show()

### Evaluating the best baseline model on validation and test data

In this step, we reload the best version of the baseline CNN and evaluate it on both the validation and test sets. This gives us the final performance values for loss, accuracy, and macro-F1, using the model that was selected during training.

To analyse the results in more detail, we generate a classification report for the test set. This provides a per-class overview of the model’s predictive performance and makes it easier to see which categories are handled well and which remain more difficult.

We also compute and visualise the confusion matrix. This is useful because it shows how predictions are distributed across the true classes and helps identify systematic confusion between specific categories.

Finally, we save the classification report, the confusion matrix, and a compact summary of the main results to disk. This makes the output easier to reuse later in the notebook and supports a clear final discussion of the baseline model.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Rebuild model and load best weights
best_baseline_model = SimpleCNN(num_classes=num_classes).to(device)
best_baseline_model.load_state_dict(
    torch.load("outputs_multiclass/baseline_cnn_best.pt", map_location=device)
)

# Evaluate on validation and test
val_loss, val_acc, val_f1, val_true, val_pred = evaluate(
    best_baseline_model, val_loader, criterion, device
)

test_loss, test_acc, test_f1, test_true, test_pred = evaluate(
    best_baseline_model, test_loader, criterion, device
)

print("Validation results:")
print(f"loss={val_loss:.4f}, acc={val_acc:.4f}, macro_f1={val_f1:.4f}")

print("\nTest results:")
print(f"loss={test_loss:.4f}, acc={test_acc:.4f}, macro_f1={test_f1:.4f}")

# Classification report for testset
test_report = classification_report(
    test_true,
    test_pred,
    target_names=classes,
    digits=4
)

print("\nClassification report (test):")
print(test_report)

# Save report
os.makedirs("outputs", exist_ok=True)
with open("outputs_multiclass/test_classification_report_baseline.txt", "w", encoding="utf-8") as f:
    f.write(test_report)

# Confusion matrix
cm = confusion_matrix(test_true, test_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Baseline CNN - Confusion Matrix (Test)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Save Confusion matrix
plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Baseline CNN - Confusion Matrix (Test)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("outputs_multiclass/test_confusion_matrix_baseline.png", dpi=150)
plt.close()

# Save summary of results and dataset info
baseline_summary = {
    "val_loss": float(val_loss),
    "val_acc": float(val_acc),
    "val_macro_f1": float(val_f1),
    "test_loss": float(test_loss),
    "test_acc": float(test_acc),
    "test_macro_f1": float(test_f1),
    "num_classes": int(num_classes),
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "test_size": int(len(test_df))
}

with open("outputs_multiclass/baseline_summary.json", "w", encoding="utf-8") as f:
    json.dump(baseline_summary, f, indent=2)

print("\Files saved:")
print("- outputs_multiclass/test_classification_report_baseline.txt")
print("- outputs_multiclass/test_confusion_matrix_baseline.png")
print("- outputs_multiclass/baseline_summary.json")

### Visualising individual predictions of the baseline model

In this step, we inspect the predictions of the trained baseline model on individual test images. To make the images readable again after normalization, we first reverse the normalization step so the visualisations correspond more closely to the original input images.

We then define a prediction function for a single image and apply it to a small set of examples spread across the test set. For each example, we show the image together with the true class, the predicted class, and the predicted probability of the selected class. This gives a more concrete view of the model’s behaviour beyond the aggregate evaluation metrics.

After that, we specifically collect and visualise misclassified test examples. This is useful because wrong predictions often reveal where the model struggles most, for example when categories are visually similar or when the model is uncertain. Such qualitative inspection helps support the final discussion of the baseline model’s strengths and weaknesses.

In [ ]:
import math
import numpy as np

# Inverse normalization for visualisatie
INV_MEAN = np.array([0.485, 0.456, 0.406])
INV_STD = np.array([0.229, 0.224, 0.225])

def denormalize_image(img_tensor):
    img = img_tensor.detach().cpu().numpy().transpose(1, 2, 0)
    img = img * INV_STD + INV_MEAN
    img = np.clip(img, 0, 1)
    return img

# Prediction function for one sample
@torch.no_grad()
def predict_single(model, image_tensor, device):
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)
    outputs = model(image_tensor)
    probs = torch.softmax(outputs, dim=1)
    pred_idx = probs.argmax(dim=1).item()
    pred_prob = probs[0, pred_idx].item()
    return pred_idx, pred_prob, probs.squeeze(0).cpu().numpy()

# Dataset for visualization (without extra augmentations)
test_vis_dataset = MinifigDataset(test_df, transform=eval_transforms)

# Choose a number of examples spread across the testset
NUM_EXAMPLES = 12
example_indices = np.linspace(0, len(test_vis_dataset) - 1, NUM_EXAMPLES, dtype=int)

examples = []
for idx in example_indices:
    image_tensor, true_label = test_vis_dataset[idx]
    pred_label, pred_prob, prob_vector = predict_single(best_baseline_model, image_tensor, device)

    examples.append({
        "idx": idx,
        "image_tensor": image_tensor,
        "true_label": true_label,
        "pred_label": pred_label,
        "pred_prob": pred_prob
    })

# General example plot
ncols = 3
nrows = math.ceil(len(examples) / ncols)

plt.figure(figsize=(15, 5 * nrows))

for i, ex in enumerate(examples, start=1):
    plt.subplot(nrows, ncols, i)

    img = denormalize_image(ex["image_tensor"])
    true_name = idx_to_class[ex["true_label"]]
    pred_name = idx_to_class[ex["pred_label"]]
    correct = ex["true_label"] == ex["pred_label"]

    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"{'✓' if correct else '✗'}\n"
        f"True: {true_name}\n"
        f"Pred: {pred_name}\n"
        f"P={ex['pred_prob']:.3f}",
        color="green" if correct else "red",
        fontsize=10
    )

plt.tight_layout()
plt.show()

# Collect all missclassified examples
misclassified = []
for idx in range(len(test_vis_dataset)):
    image_tensor, true_label = test_vis_dataset[idx]
    pred_label, pred_prob, _ = predict_single(best_baseline_model, image_tensor, device)

    if pred_label != true_label:
        misclassified.append((idx, image_tensor, true_label, pred_label, pred_prob))

print(f"Amount of wrongfully classified test examples: {len(misclassified)}")

# Show a selection of misclassified examples
NUM_ERRORS_TO_SHOW = min(9, len(misclassified))

if NUM_ERRORS_TO_SHOW > 0:
    plt.figure(figsize=(15, 12))

    for i in range(NUM_ERRORS_TO_SHOW):
        idx, image_tensor, true_label, pred_label, pred_prob = misclassified[i]

        plt.subplot(3, 3, i + 1)
        img = denormalize_image(image_tensor)

        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"True: {idx_to_class[true_label]}\n"
            f"Pred: {idx_to_class[pred_label]}\n"
            f"P={pred_prob:.3f}",
            color="red",
            fontsize=10
        )

    plt.tight_layout()
    plt.show()

### Analysing baseline performance at the class level

In this step, we move from overall test performance to a more detailed class-level analysis. We convert the classification report into a structured table so that the precision, recall, F1-score, and support can be inspected separately for each category.

We then identify the best and worst performing classes based on their F1-scores. This makes it easier to see which categories the baseline model can recognise well and which categories remain difficult to classify. Saving these per-class metrics to a CSV file also makes the results easier to reuse in later comparisons.

Finally, we visualise the strongest and weakest classes and explore the relation between class support and F1-score. This helps us assess whether some of the weaker results may be linked to the number of available examples per class, which is relevant when interpreting the limitations of the baseline model.

In [ ]:
# Classification report as a dictionary
report_dict = classification_report(
    test_true,
    test_pred,
    target_names=classes,
    output_dict=True,
    zero_division=0
)

# Convert to DataFrame
report_df = pd.DataFrame(report_dict).transpose()

# Only keep real classes, not the summary rows
class_report_df = report_df.loc[classes].copy()

# Support as integer
class_report_df["support"] = class_report_df["support"].astype(int)

# Sort on F1-score
best_classes = class_report_df.sort_values("f1-score", ascending=False).head(15)
worst_classes = class_report_df.sort_values("f1-score", ascending=True).head(15)

print("Top 15 worst scoring classes on test F1:")
print(best_classes[["precision", "recall", "f1-score", "support"]])

print("\nTop 15 worst scoring classes on test F1:")
print(worst_classes[["precision", "recall", "f1-score", "support"]])

# Save per-class metrics to CSV
class_report_df.to_csv("outputs_multiclass/test_per_class_metrics_baseline.csv", index=True)

# Plot best scoring classes
plt.figure(figsize=(10, 6))
plt.barh(best_classes.index[::-1], best_classes["f1-score"][::-1])
plt.xlabel("F1-score")
plt.title("Top 15 best scoring classes (test)")
plt.tight_layout()
plt.show()

# Plot worst scoring classes
plt.figure(figsize=(10, 6))
plt.barh(worst_classes.index[::-1], worst_classes["f1-score"][::-1])
plt.xlabel("F1-score")
plt.title("Top 15 worst scoring classes (test)")
plt.tight_layout()
plt.show()

# Extra analysis: relation between support and F1
plt.figure(figsize=(8, 6))
plt.scatter(class_report_df["support"], class_report_df["f1-score"])
plt.xlabel("Support (number of test examples)")
plt.ylabel("F1-score")
plt.title("Relation between support and F1-score per class")
plt.tight_layout()
plt.show()

print("\Files saved:")
print("- outputs_multiclass/test_per_class_metrics_baseline.csv")

### Conclusion

The baseline CNN provides a useful first reference point for the multiclass classification task. It shows that category prediction from minifig images is feasible, which means the model is able to learn relevant visual patterns from the data. At the same time, the results also make clear that this is a challenging problem, especially because the dataset contains many categories that can still look visually similar.

The main weakness of the baseline model is that its performance is not equally strong across all classes. More distinctive categories are generally easier to recognise, while smaller or more visually overlapping categories remain more difficult. This suggests that the model captures broad visual cues, but still struggles with finer distinctions between themes.

Overall, the baseline is a solid starting point for the assignment, but it is not yet strong enough to be considered an optimal solution. Its main value lies in establishing a clear reference against which a more advanced pretrained model can later be compared.

# Part 1.2 - Using a pretrained CNN, EfficientNet
### Defining the EfficientNet-B0 transfer learning model

In this step, we replace the baseline CNN with a pretrained EfficientNet-B0 model. Instead of training the full network from scratch, we start from weights that were learned on a large image dataset and adapt the model to our own multiclass task. This is a standard transfer learning approach and is especially useful when working with a more limited task-specific dataset.

To do this, we freeze the convolutional feature extractor and only replace the final classifier layer so that the model outputs the correct number of category classes. This means that the pretrained visual features are kept fixed, while the new classification head is trained specifically for the Lego Minifigs dataset.

We then define the loss function and optimizer for this model, making sure that only the trainable parameters are updated. Finally, we check how many parameters remain trainable and run a small forward-pass sanity check on one batch. This confirms that the adapted EfficientNet produces outputs with the expected shape before training starts.

In [ ]:
# Build EfficientNet-B0 with pretrained weights
effnet_model = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

# Freeze backbone 
for param in effnet_model.features.parameters():
    param.requires_grad = False

# Change classifier
in_features = effnet_model.classifier[1].in_features
effnet_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_classes)
)

# To device
effnet_model = effnet_model.to(device)

# Loss and optimizer
effnet_criterion = nn.CrossEntropyLoss()
effnet_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, effnet_model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

print(effnet_model)

# Controle: how many parameters are trainable?
total_params = sum(p.numel() for p in effnet_model.parameters())
trainable_params = sum(p.numel() for p in effnet_model.parameters() if p.requires_grad)

print(f"\nTotal number of parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Sanity check on one batch
effnet_model.eval()
with torch.no_grad():
    sample_outputs = effnet_model(images.to(device))

print("\nOutput shape:", sample_outputs.shape)
print("Expected shape:", (images.shape[0], num_classes))

### Training EfficientNet-B0 in the first transfer learning phase

In this step, we train the adapted EfficientNet-B0 model while keeping the pretrained feature extractor frozen. This means that only the new classification head is updated during training. The goal of this first phase is to let the model adjust its final decision layer to the Lego Minifigs categories without immediately changing the pretrained visual features.

As with the baseline model, we store the full training history, save it to disk, and identify the best epoch based on the validation macro-F1 score. This makes it possible to track how the model evolves during training and to select the best-performing version in a consistent way.

Finally, we visualise the training and validation curves for loss, accuracy, and macro-F1. These plots help us assess whether the classifier head is learning effectively and whether the transfer learning setup already provides an improvement over the baseline model.

In [ ]:
# Make sure that the outputs directory exists
os.makedirs("outputs", exist_ok=True)

# Fase 1 training: only train classifier
effnet_history_phase1 = train_model(
    model=effnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=effnet_criterion,
    optimizer=effnet_optimizer,
    device=device,
    epochs=12,
    patience=4,
    model_path="outputs_multiclass/effnet_b0_phase1_best.pt"
)

print("\nTrainingshistory EfficientNet fase 1:")
print(effnet_history_phase1)

# Save history to CSV
effnet_history_phase1.to_csv("outputs_multiclass/effnet_b0_phase1_history.csv", index=False)
print("\nHistoriek saved as outputs_multiclass/effnet_b0_phase1_history.csv")

# Beste epoch given validation macro-F1
best_idx = effnet_history_phase1["val_f1"].idxmax()
best_row = effnet_history_phase1.loc[best_idx]

print("\nBeste epoch given validation macro-F1 (fase 1):")
print(best_row)

# Plot 1: loss
plt.figure(figsize=(8, 5))
plt.plot(effnet_history_phase1["epoch"], effnet_history_phase1["train_loss"], label="Train loss")
plt.plot(effnet_history_phase1["epoch"], effnet_history_phase1["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("EfficientNet-B0 Phase 1 - Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: accuracy
plt.figure(figsize=(8, 5))
plt.plot(effnet_history_phase1["epoch"], effnet_history_phase1["train_acc"], label="Train accuracy")
plt.plot(effnet_history_phase1["epoch"], effnet_history_phase1["val_acc"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("EfficientNet-B0 Phase 1 - Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# Plot 3: macro-F1
plt.figure(figsize=(8, 5))
plt.plot(effnet_history_phase1["epoch"], effnet_history_phase1["train_f1"], label="Train macro-F1")
plt.plot(effnet_history_phase1["epoch"], effnet_history_phase1["val_f1"], label="Validation macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("EfficientNet-B0 Phase 1 - Macro-F1")
plt.legend()
plt.tight_layout()
plt.show()

### Evaluating the best EfficientNet-B0 model from phase 1

In this step, we rebuild the EfficientNet-B0 architecture, load the best saved weights from the first training phase, and evaluate the model on both the validation and test sets. This gives us the final phase 1 performance values for loss, accuracy, and macro-F1, based on the model version that performed best on the validation data.

To examine the results in more detail, we generate a classification report for the test set. This makes it possible to inspect the performance of the model at the class level and to see which categories are predicted well and which remain more difficult.

We also compute and visualise the confusion matrix. This helps identify which classes are most often confused with each other and gives a clearer picture of the types of errors the model still makes.

Finally, we save the classification report, confusion matrix, and a summary of the main metrics to disk. This ensures that the EfficientNet results are documented in the same structured way as the baseline, which makes the later comparison between both models more straightforward.

In [ ]:
# Rebuild best EfficientNet fase 1 model and load best weights
best_effnet_phase1 = models.efficientnet_b0(
    weights=None
)

# Same classifier as during training
in_features = best_effnet_phase1.classifier[1].in_features
best_effnet_phase1.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_classes)
)

best_effnet_phase1 = best_effnet_phase1.to(device)
best_effnet_phase1.load_state_dict(
    torch.load("outputs_multiclass/effnet_b0_phase1_best.pt", map_location=device)
)

# Evaluatie on validation and test
eff_val_loss, eff_val_acc, eff_val_f1, eff_val_true, eff_val_pred = evaluate(
    best_effnet_phase1, val_loader, effnet_criterion, device
)

eff_test_loss, eff_test_acc, eff_test_f1, eff_test_true, eff_test_pred = evaluate(
    best_effnet_phase1, test_loader, effnet_criterion, device
)

print("EfficientNet Phase 1 - Validation results:")
print(f"loss={eff_val_loss:.4f}, acc={eff_val_acc:.4f}, macro_f1={eff_val_f1:.4f}")

print("\nEfficientNet Phase 1 - Test results:")
print(f"loss={eff_test_loss:.4f}, acc={eff_test_acc:.4f}, macro_f1={eff_test_f1:.4f}")

# Classification report
eff_test_report = classification_report(
    eff_test_true,
    eff_test_pred,
    target_names=classes,
    digits=4,
    zero_division=0
)

print("\nClassification report (test):")
print(eff_test_report)

with open("outputs_multiclass/test_classification_report_effnet_phase1.txt", "w", encoding="utf-8") as f:
    f.write(eff_test_report)

# Confusion matrix
eff_cm = confusion_matrix(eff_test_true, eff_test_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    eff_cm,
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("EfficientNet-B0 Phase 1 - Confusion Matrix (Test)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 12))
sns.heatmap(
    eff_cm,
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("EfficientNet-B0 Phase 1 - Confusion Matrix (Test)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("outputs_multiclass/test_confusion_matrix_effnet_phase1.png", dpi=150)
plt.close()

# Save summary
effnet_phase1_summary = {
    "val_loss": float(eff_val_loss),
    "val_acc": float(eff_val_acc),
    "val_macro_f1": float(eff_val_f1),
    "test_loss": float(eff_test_loss),
    "test_acc": float(eff_test_acc),
    "test_macro_f1": float(eff_test_f1)
}

with open("outputs_multiclass/effnet_phase1_summary.json", "w", encoding="utf-8") as f:
    json.dump(effnet_phase1_summary, f, indent=2)

print("\nFiles saved:")
print("- outputs_multiclass/test_classification_report_effnet_phase1.txt")
print("- outputs_multiclass/test_confusion_matrix_effnet_phase1.png")
print("- outputs_multiclass/effnet_phase1_summary.json")

### Comparing the baseline CNN with EfficientNet-B0

In this final comparison step, we place the baseline CNN and EfficientNet-B0 side by side using the main validation and test metrics. This makes it possible to compare both models in a direct and structured way, using the same evaluation criteria for each approach.

We then calculate the difference in test accuracy and test macro-F1 between the two models. These differences show more clearly whether EfficientNet-B0 provides a meaningful improvement over the baseline rather than only a small numerical change.

Finally, we visualise the test accuracy and test macro-F1 of both models with simple bar plots. This helps make the comparison easier to interpret and supports the final discussion of whether transfer learning leads to a better multiclass classification model for the Lego Minifigs dataset.

In [ ]:
# Summarising table for comparison
comparison_df = pd.DataFrame([
    {
        "model": "Baseline CNN",
        "val_acc": val_acc,
        "val_macro_f1": val_f1,
        "test_acc": test_acc,
        "test_macro_f1": test_f1
    },
    {
        "model": "EfficientNet-B0 Phase 1",
        "val_acc": eff_val_acc,
        "val_macro_f1": eff_val_f1,
        "test_acc": eff_test_acc,
        "test_macro_f1": eff_test_f1
    }
])

print("Model comparison:")
print(comparison_df)

comparison_df.to_csv("outputs_multiclass/model_comparison.csv", index=False)

# Explicitly calculate the improvements
acc_diff = eff_test_acc - test_acc
f1_diff = eff_test_f1 - test_f1

print("\nImprovement of EfficientNet t.o.v. baseline on testset:")
print(f"Accuracy difference: {acc_diff:.4f}")
print(f"Macro-F1 difference: {f1_diff:.4f}")

# Plot 1: test accuracy
plt.figure(figsize=(8, 5))
plt.bar(comparison_df["model"], comparison_df["test_acc"])
plt.ylabel("Test accuracy")
plt.title("Comparison of test accuracy")
plt.tight_layout()
plt.show()

# Plot 2: test macro-F1
plt.figure(figsize=(8, 5))
plt.bar(comparison_df["model"], comparison_df["test_macro_f1"])
plt.ylabel("Test macro-F1")
plt.title("Comparison of test macro-F1")
plt.tight_layout()
plt.show()

## Final conclusion

In this multiclass task, we compared a baseline CNN trained from scratch with a pretrained EfficientNet-B0 model in a first transfer learning setup. Both models were evaluated using validation and test accuracy as well as macro-F1, so that we could assess not only overall correctness but also how well the models handled the different classes in a balanced way.

The comparison shows that the pretrained EfficientNet-B0 model performs at least on the same level as the baseline in terms of overall accuracy, while achieving a clearer improvement in macro-F1. This suggests that transfer learning is especially beneficial for the class-balanced aspect of the problem. In practice, this means that the pretrained model is better able to deal with the multiclass structure of the dataset and offers a stronger overall solution than the baseline CNN.

Overall, this task confirms that image-based category prediction for Lego Minifigs is feasible, but also that performance depends strongly on the representational strength of the model. The baseline CNN provides a useful reference, while EfficientNet-B0 shows that a pretrained architecture is a more effective choice for this multiclass classification problem.

# Part 2 — Multilabel classification

In this part of the notebook, we move from multiclass classification to a multilabel image classification task. Instead of predicting exactly one category for each Lego Minifig, the goal is now to predict which themes are present for each image. This makes the problem more challenging, because a single minifig can belong to multiple themes at the same time.

The workflow remains similar to the multiclass part of the notebook: we first prepare and inspect the data, then construct the target representation, build the image pipeline, train the models, and finally evaluate the results. The key difference is that the target is no longer a single class label, but a set of labels per image. As a result, both the output representation and the evaluation approach have to be adapted to the multilabel setting.

In the following section, we therefore focus on preparing the theme labels correctly and setting up a pipeline that can handle multilabel predictions in a consistent and interpretable way.

### Loading the data for the multilabel task

In this step, we load the Lego Minifigs metadata again and convert it into a pandas DataFrame for the multilabel part of the notebook. We then perform a first cleaning step by removing observations without a valid image path, construct the full local image paths, and keep only the observations for which the corresponding image file is available.

After that, we take a first look at the `themes` column, since this will be the target structure for the multilabel task. Unlike the multiclass setting, each image can now be linked to multiple labels, so it is important to inspect this column before encoding the targets.

Finally, we prepare the computation device and define the image transforms that will be used later in the training and evaluation pipeline.

In [ ]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
df = df.dropna(subset=["img_local_path"]).copy()

df["full_path"] = df["img_local_path"].apply(build_full_path)
df["file_exists"] = df["full_path"].apply(os.path.exists)
df = df[df["file_exists"]].copy()

print("Number of rows after cleaning and file check:", len(df))
print("\nExample values from the themes column:")
print(df["themes"].head())

### Exploring and selecting the multilabel target themes

In this step, we prepare the `themes` column for the multilabel task. We first ensure that every observation contains a proper Python list, so that the theme information can be processed consistently. After that, we flatten all theme labels across the dataset and count how often each theme occurs.

This allows us to inspect the multilabel target distribution in detail. We look at the most frequent themes, summarise the frequency distribution, and examine several thresholds to see how many themes remain above different minimum support levels. We also visualise the most common themes to get a clearer overview of the label distribution.

Based on this exploration, we define a shortlist of selected themes by keeping only labels with at least a minimum number of examples. We then remove observations that do not contain any of these selected themes. This creates a cleaner multilabel classification problem, where the retained labels have enough support to be learned more reliably by the model.

In [ ]:
# Make sure that all values in the themes column are lists (some might be NaN or other types)
def ensure_list(x):
    if isinstance(x, list):
        return x
    return []

df["themes_list"] = df["themes"].apply(ensure_list)

# Flatten all themes
all_themes = [theme for theme_list in df["themes_list"] for theme in theme_list]
theme_counts = pd.Series(all_themes).value_counts()

print("Number of unique themes:", len(theme_counts))
print("\nTop 30 most common themes:")
print(theme_counts.head(30))

# Descriptive statistics of theme frequencies
print("\nDescriptive statistics of theme frequencies:")
print(theme_counts.describe())

# Define thresholds
thresholds = [5, 10, 20, 50, 100]
print()
for t in thresholds:
    n_themes = (theme_counts >= t).sum()
    print(f"Number of themes with at least {t} examples: {n_themes}")

# Visualisation for top 20 themes
plt.figure(figsize=(12, 6))
sns.barplot(x=theme_counts.index[:20], y=theme_counts.values[:20])
plt.xticks(rotation=90)
plt.title("Top 20 most common themes")
plt.xlabel("Theme")
plt.ylabel("Number of examples")
plt.tight_layout()
plt.show()

# Definitive shortlist
MIN_SAMPLES_PER_THEME = 50
selected_themes = theme_counts[theme_counts >= MIN_SAMPLES_PER_THEME].index.tolist()

print("\nSelected themes (at least 50 examples):")
print(selected_themes)

print("\nNumber of selected themes:", len(selected_themes))

# Only keep minifigs that have at least one selected theme
df["selected_themes"] = df["themes_list"].apply(
    lambda lst: [theme for theme in lst if theme in selected_themes]
)

df_multi = df[df["selected_themes"].apply(len) > 0].copy()

print("\nNumber of rows before multilabel-filter:", len(df))
print("Number of rows after multilabel-filter:", len(df_multi))
print(f"Percentage behouden: {100 * len(df_multi) / len(df):.2f}%")

# Example control
print("\nExample of original themes and selected themes:")
print(df_multi[["themes_list", "selected_themes"]].head(10))

### Encoding the multilabel targets

In this step, we convert the selected theme labels into a machine-readable multilabel representation. We first create mappings between theme names and numeric indices, so that each selected theme receives a fixed position in the output space.

Next, we transform the list of selected themes for each minifig into a multi-hot target vector. In this representation, each position corresponds to one theme: a value of 1 indicates that the theme is present, while a value of 0 indicates that it is absent. This is the standard target format for multilabel classification and is required for model training.

We then inspect a few examples to verify that the encoding works correctly and create a compact modeling DataFrame containing the most relevant fields for the next steps. Finally, we analyse how many labels each minifig has on average. This is useful because it gives a clearer picture of the complexity of the multilabel task and the structure of the final target space.

In [ ]:
# Mapping between themes and indices
theme_to_idx = {theme: idx for idx, theme in enumerate(selected_themes)}
idx_to_theme = {idx: theme for theme, idx in theme_to_idx.items()}

print("Number of multilabel-targets:", len(selected_themes))
print("\nExample mapping:")
print(list(theme_to_idx.items())[:15])

# Function to create multi-hot vector
def make_multihot(theme_list, theme_to_idx):
    vector = np.zeros(len(theme_to_idx), dtype=np.float32)
    for theme in theme_list:
        if theme in theme_to_idx:
            vector[theme_to_idx[theme]] = 1.0
    return vector

df_multi["target_vector"] = df_multi["selected_themes"].apply(
    lambda lst: make_multihot(lst, theme_to_idx)
)

# Control of enkele voorbeelden
print("\nExample of selected themes and target vector:")
for i in range(5):
    row = df_multi.iloc[i]
    print(f"Selected themes: {row['selected_themes']}")
    print(f"Target vector sum: {int(row['target_vector'].sum())}")
    print(f"Target vector shape: {row['target_vector'].shape}")
    print("-" * 50)

# Compact modeling dataframe
multi_model_df = df_multi[["id", "name", "full_path", "selected_themes", "target_vector"]].copy()

print("\nShape of multilabel-modeldataframe:", multi_model_df.shape)

# Analysis: how many labels per minifig?
multi_model_df["num_labels"] = multi_model_df["target_vector"].apply(lambda x: int(x.sum()))

print("\nNumber of labels per minifig:")
print(multi_model_df["num_labels"].value_counts().sort_index())

plt.figure(figsize=(8, 5))
multi_model_df["num_labels"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Number of labels per minifig")
plt.ylabel("Number of minifigs")
plt.title("Distribution of number of multilabels per minifig")
plt.tight_layout()
plt.show()

### Creating and checking the train, validation, and test split for the multilabel task

In this step, we divide the multilabel dataset into training, validation, and test sets. We first create a hold-out test set and then split the remaining data again to obtain a validation set. This gives us a consistent setup for model training, model selection, and final evaluation.

Because this is a multilabel problem, it is not enough to only check the subset sizes. We also compare the average label prevalence across the train, validation, and test sets. In other words, we verify how often each theme is active in each subset. This is an important consistency check, because the subsets should remain as comparable as possible in terms of label distribution.

In addition, we inspect the average number of labels per minifig in each subset. This helps confirm that the multilabel structure is distributed in a similar way across train, validation, and test data. Finally, we visualise the label prevalence of the most common themes and save the resulting splits to disk for later reuse in the modeling pipeline.

In [ ]:
# First split train+val and test
train_val_df, test_df = train_test_split(
    multi_model_df,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

# After that, split train and validation
val_relative_size = 0.15 / 0.85

train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_relative_size,
    random_state=42,
    shuffle=True
)

# Basic control
total = len(multi_model_df)

print("Split sizes:")
print(f"Train: {len(train_df)} ({len(train_df)/total:.4f})")
print(f"Val:   {len(val_df)} ({len(val_df)/total:.4f})")
print(f"Test:  {len(test_df)} ({len(test_df)/total:.4f})")

# Helper to calculate average label activation per theme
def label_prevalence(df_part):
    matrix = np.stack(df_part["target_vector"].values)
    return matrix.mean(axis=0)

train_prev = label_prevalence(train_df)
val_prev = label_prevalence(val_df)
test_prev = label_prevalence(test_df)

prevalence_df = pd.DataFrame({
    "theme": selected_themes,
    "train_prev": train_prev,
    "val_prev": val_prev,
    "test_prev": test_prev
})

print("\nExample of labelprevalence per theme:")
print(prevalence_df.head(15))

# Average absolute differences between subsets
train_val_diff = np.mean(np.abs(prevalence_df["train_prev"] - prevalence_df["val_prev"]))
train_test_diff = np.mean(np.abs(prevalence_df["train_prev"] - prevalence_df["test_prev"]))

print("\nAverage absolute difference in labelprevalence:")
print(f"Train vs Val:  {train_val_diff:.6f}")
print(f"Train vs Test: {train_test_diff:.6f}")

# Number of labels per minifig per subset
print("\nAverage number of labels per minifig:")
print(f"Train: {train_df['num_labels'].mean():.4f}")
print(f"Val:   {val_df['num_labels'].mean():.4f}")
print(f"Test:  {test_df['num_labels'].mean():.4f}")

# Visualisation of prevalence on top 20 most prevalent selected themes
top_themes = theme_counts.loc[selected_themes].sort_values(ascending=False).head(20).index.tolist()
plot_df = prevalence_df.set_index("theme").loc[top_themes]

plt.figure(figsize=(12, 6))
x = np.arange(len(plot_df))
width = 0.25

plt.bar(x - width, plot_df["train_prev"], width=width, label="Train")
plt.bar(x,         plot_df["val_prev"],   width=width, label="Validation")
plt.bar(x + width, plot_df["test_prev"],  width=width, label="Test")

plt.xticks(x, plot_df.index, rotation=90)
plt.ylabel("average label activation")
plt.title("Labelprevalence per subset (top 20 themes)")
plt.legend()
plt.tight_layout()
plt.show()

# Splits opslaan
os.makedirs("outputs_multilabel", exist_ok=True)

train_df.to_pickle("outputs_multilabel/train_split_multilabel.pkl")
val_df.to_pickle("outputs_multilabel/val_split_multilabel.pkl")
test_df.to_pickle("outputs_multilabel/test_split_multilabel.pkl")

print("\nSplits saved in outputs_multilabel/")

### Building the multilabel dataset and dataloaders

In this step, we adapt the dataset class to the multilabel setting. Instead of returning a single class index for each image, the dataset now returns a multi-hot target vector that represents all active themes for that minifig. This is necessary because each image can belong to multiple labels at the same time.

We then create separate datasets for the training, validation, and test sets, using the appropriate image transforms for each subset. After that, we build dataloaders that organise the data into batches, which makes model training and evaluation more efficient.

Finally, we run several sanity checks on the output of the data pipeline. We inspect the batch shapes, verify the datatype of the targets, check how many labels are active for a few examples, and confirm the shape of one individual sample. This ensures that the multilabel targets are correctly formatted before the model is defined and trained.

In [ ]:
# Adapted Dataset for multilabel targets
class MinifigMultilabelDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["full_path"]).convert("RGB")
        target = torch.tensor(row["target_vector"], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, target

# Make datasets
train_dataset = MinifigMultilabelDataset(train_df, transform=train_transforms)
val_dataset = MinifigMultilabelDataset(val_df, transform=eval_transforms)
test_dataset = MinifigMultilabelDataset(test_df, transform=eval_transforms)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

# Make dataloaders
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# Sanity check on one batch
images, targets = next(iter(train_loader))

print("\nImage batch shape:", images.shape)
print("Target batch shape:", targets.shape)
print("Target dtype:", targets.dtype)

# Control of target values
print("\nNumber of active labels in first 10 examples of the batch:")
print(targets[:10].sum(dim=1))

# Control of one sample
sample_image, sample_target = train_dataset[0]
print("\nSingle image tensor shape:", sample_image.shape)
print("Single target shape:", sample_target.shape)
print("Number of active labels in sample:", int(sample_target.sum().item()))

# Number of output labels for the model
num_labels = len(selected_themes)
print("\nNumber of multilabel outputs:", num_labels)

### Defining the baseline multilabel CNN and training procedure

In this step, we define a baseline convolutional neural network for the multilabel classification task. The model extracts visual features from the input images through several convolutional blocks and then maps these features to a set of output labels, where each output corresponds to one possible theme.

Because this is a multilabel problem, the model does not predict a single class per image. Instead, it produces one score per label, and these scores are converted into probabilities with a sigmoid function. A fixed decision threshold is then used to determine which labels are predicted as active. This differs from the multiclass setting, where only one class can be selected.

We also define the multilabel loss function, the optimizer, and separate functions for training and evaluation. During evaluation, we report both micro-F1 and macro-F1, since these metrics provide complementary information about multilabel performance. The training loop uses early stopping based on validation macro-F1, so that the best model is selected according to balanced label-level performance.

Finally, we perform a small sanity check on one batch to verify that the model outputs the expected number of label scores and that the predicted probabilities have the correct structure before training begins.

In [ ]:
# Baseline CNN for multilabel
class SimpleMultilabelCNN(nn.Module):
    def __init__(self, num_labels):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 224 -> 112

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 112 -> 56

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 56 -> 28

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)    # 28 -> 14
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_labels)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

# Model, loss and optimizer
multilabel_model = SimpleMultilabelCNN(num_labels=num_labels).to(device)
multilabel_criterion = nn.BCEWithLogitsLoss()
multilabel_optimizer = torch.optim.Adam(
    multilabel_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print(multilabel_model)

# Helper: logits -> binairy predictions
def logits_to_predictions(logits, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    return preds

# Train one epoch
def train_one_epoch_multilabel(model, loader, criterion, optimizer, device, threshold=0.5):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_targets = []

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = logits_to_predictions(logits, threshold=threshold)
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(targets.detach().cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    epoch_loss = running_loss / len(loader.dataset)
    micro_f1 = f1_score(all_targets, all_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)

    return epoch_loss, micro_f1, macro_f1

# Evaluation
@torch.no_grad()
def evaluate_multilabel(model, loader, criterion, device, threshold=0.5):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    all_probs = []

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = criterion(logits, targets)

        running_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()

        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(targets.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    all_probs = np.vstack(all_probs)

    epoch_loss = running_loss / len(loader.dataset)
    micro_f1 = f1_score(all_targets, all_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)

    return epoch_loss, micro_f1, macro_f1, all_targets, all_preds, all_probs

# Training loop using early stopping on validation macro-F1
def train_model_multilabel(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=20,
    patience=5,
    threshold=0.5,
    model_path="outputs_multilabel/best_multilabel_model.pt"
):
    best_macro_f1 = -1.0
    patience_counter = 0
    history = []

    for epoch in range(epochs):
        train_loss, train_micro_f1, train_macro_f1 = train_one_epoch_multilabel(
            model, train_loader, criterion, optimizer, device, threshold=threshold
        )

        val_loss, val_micro_f1, val_macro_f1, _, _, _ = evaluate_multilabel(
            model, val_loader, criterion, device, threshold=threshold
        )

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_micro_f1": train_micro_f1,
            "train_macro_f1": train_macro_f1,
            "val_loss": val_loss,
            "val_micro_f1": val_micro_f1,
            "val_macro_f1": val_macro_f1
        }
        history.append(row)

        print(
            f"Epoch {epoch+1:02d} | "
            f"train_loss={train_loss:.4f} | train_micro_f1={train_micro_f1:.4f} | train_macro_f1={train_macro_f1:.4f} | "
            f"val_loss={val_loss:.4f} | val_micro_f1={val_micro_f1:.4f} | val_macro_f1={val_macro_f1:.4f}"
        )

        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            patience_counter = 0
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    history_df = pd.DataFrame(history)
    return history_df

# Sanity check: forward pass
multilabel_model.eval()
with torch.no_grad():
    sample_logits = multilabel_model(images.to(device))

print("\nOutput shape:", sample_logits.shape)
print("Expected shape:", (images.shape[0], num_labels))

sample_probs = torch.sigmoid(sample_logits[:2]).cpu()
print("\nExample of sigmoid probabilities for first 2 examples:")
print(sample_probs[:, :10])

### Training the baseline multilabel model

In this step, we train the baseline multilabel CNN on the training set and evaluate it after each epoch on the validation set. The full training history is stored so that we can analyse how the model evolves over time and determine which epoch gives the best validation macro-F1 score.

After training, we save the history to disk and identify the best epoch based on validation macro-F1, since this is the metric used for model selection in the training loop. This gives us a clear reference point for the strongest version of the baseline multilabel model.

Finally, we visualise the learning process with plots for loss, micro-F1, and macro-F1 on both the training and validation sets. These figures help us assess whether the model is learning effectively and whether there are signs of underfitting or overfitting during training.

In [ ]:
# Train the baseline multilabel-model
multilabel_history = train_model_multilabel(
    model=multilabel_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=multilabel_criterion,
    optimizer=multilabel_optimizer,
    device=device,
    epochs=20,
    patience=5,
    threshold=0.5,
    model_path="outputs_multilabel/baseline_multilabel_best.pt"
)

print("\nTraining history for multilabel baseline:")
print(multilabel_history)

# Saving history to CSV
multilabel_history.to_csv("outputs_multilabel/baseline_multilabel_history.csv", index=False)
print("\nHistory saved as outputs_multilabel/baseline_multilabel_history.csv")

# Deciding the best epoch
best_idx = multilabel_history["val_macro_f1"].idxmax()
best_row = multilabel_history.loc[best_idx]

print("\nBest epoch on validation macro-F1:")
print(best_row)

# Plot 1: loss
plt.figure(figsize=(8, 5))
plt.plot(multilabel_history["epoch"], multilabel_history["train_loss"], label="Train loss")
plt.plot(multilabel_history["epoch"], multilabel_history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline Multilabel CNN - Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: micro-F1
plt.figure(figsize=(8, 5))
plt.plot(multilabel_history["epoch"], multilabel_history["train_micro_f1"], label="Train micro-F1")
plt.plot(multilabel_history["epoch"], multilabel_history["val_micro_f1"], label="Validation micro-F1")
plt.xlabel("Epoch")
plt.ylabel("Micro-F1")
plt.title("Baseline Multilabel CNN - Micro-F1")
plt.legend()
plt.tight_layout()
plt.show()

# Plot 3: macro-F1
plt.figure(figsize=(8, 5))
plt.plot(multilabel_history["epoch"], multilabel_history["train_macro_f1"], label="Train macro-F1")
plt.plot(multilabel_history["epoch"], multilabel_history["val_macro_f1"], label="Validation macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Baseline Multilabel CNN - Macro-F1")
plt.legend()
plt.tight_layout()
plt.show()

### Tuning the decision threshold for multilabel prediction

In this step, we reload the best baseline multilabel model and collect its predicted probabilities on the validation set. Instead of immediately converting these probabilities into binary predictions with a fixed threshold of 0.5, we first evaluate a range of possible thresholds.

This is an important step in multilabel classification, because the final predictions depend strongly on the threshold used to decide whether a label is active or not. A threshold that is too high may miss relevant labels, while a threshold that is too low may predict too many labels per image. We therefore compare several threshold values using validation micro-F1, validation macro-F1, and the average number of predicted labels per example.

Based on these results, we select the threshold that gives the best validation macro-F1 score. We also save the threshold tuning results and visualise how the validation performance changes across thresholds. This allows us to choose a better decision rule before evaluating the final model on the test set.

In [ ]:
# Reload best model
best_multilabel_model = SimpleMultilabelCNN(num_labels=num_labels).to(device)
best_multilabel_model.load_state_dict(
    torch.load("outputs_multilabel/baseline_multilabel_best.pt", map_location=device)
)

# First retreive probabilities on validation set, without thresholding
@torch.no_grad()
def collect_probs_and_targets(model, loader, device):
    model.eval()
    all_probs = []
    all_targets = []

    for images, targets in loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_targets.append(targets.numpy())

    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)
    return all_probs, all_targets

val_probs, val_targets = collect_probs_and_targets(best_multilabel_model, val_loader, device)

# Threshold grid
thresholds = [0.01, 0.02, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

rows = []
for threshold in thresholds:
    val_preds = (val_probs >= threshold).astype(np.float32)

    micro_f1 = f1_score(val_targets, val_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)
    avg_predicted_labels = val_preds.sum(axis=1).mean()

    rows.append({
        "threshold": threshold,
        "val_micro_f1": micro_f1,
        "val_macro_f1": macro_f1,
        "avg_predicted_labels_per_example": avg_predicted_labels
    })

threshold_results = pd.DataFrame(rows)

print("Threshold tuning results:")
print(threshold_results)

# Choose best threshold based on validation macro-F1
best_idx = threshold_results["val_macro_f1"].idxmax()
best_threshold_row = threshold_results.loc[best_idx]
best_threshold = float(best_threshold_row["threshold"])

print("\nBest threshold on validation macro-F1:")
print(best_threshold_row)

threshold_results.to_csv("outputs_multilabel/threshold_tuning_results.csv", index=False)

# Plot micro-F1
plt.figure(figsize=(8, 5))
plt.plot(threshold_results["threshold"], threshold_results["val_micro_f1"], marker="o")
plt.xlabel("Threshold")
plt.ylabel("Validation micro-F1")
plt.title("Threshold tuning - validation micro-F1")
plt.tight_layout()
plt.show()

# Plot macro-F1
plt.figure(figsize=(8, 5))
plt.plot(threshold_results["threshold"], threshold_results["val_macro_f1"], marker="o")
plt.xlabel("Threshold")
plt.ylabel("Validation macro-F1")
plt.title("Threshold tuning - validation macro-F1")
plt.tight_layout()
plt.show()

# Plot number of predicted labels
plt.figure(figsize=(8, 5))
plt.plot(threshold_results["threshold"], threshold_results["avg_predicted_labels_per_example"], marker="o")
plt.xlabel("Threshold")
plt.ylabel("Average number of predicted labels")
plt.title("Threshold tuning - predicted labels per example")
plt.tight_layout()
plt.show()

print("\nResults saved as:")
print("- outputs_multilabel/threshold_tuning_results.csv")
print(f"\nChosen threshold: {best_threshold:.4f}")

### Evaluating the baseline multilabel model on the test set

In this step, we evaluate the best baseline multilabel model on the hold-out test set using the threshold selected from the validation analysis. This gives us the final test performance of the model in terms of loss, micro-F1, and macro-F1.

We also compare the average number of true labels and predicted labels per example. This is useful in multilabel classification, because it shows whether the model tends to predict too many labels, too few labels, or roughly the right amount.

In addition, we calculate the F1-score separately for each theme and rank the themes from best to worst. This provides a more detailed view of the model’s behaviour across individual labels and helps identify which themes are learned well and which remain difficult. Finally, we save both the per-theme results and a compact summary of the main test metrics to disk.

In [ ]:
# Chosen threshold based on validation-analysis
best_threshold = 0.08

# Reload best model
best_multilabel_model = SimpleMultilabelCNN(num_labels=num_labels).to(device)
best_multilabel_model.load_state_dict(
    torch.load("outputs_multilabel/baseline_multilabel_best.pt", map_location=device)
)

# Test evaluation
test_loss, test_micro_f1, test_macro_f1, test_targets, test_preds, test_probs = evaluate_multilabel(
    best_multilabel_model,
    test_loader,
    multilabel_criterion,
    device,
    threshold=best_threshold
)

print("Multilabel baseline - Test results:")
print(f"Threshold: {best_threshold:.2f}")
print(f"loss={test_loss:.4f}")
print(f"micro_f1={test_micro_f1:.4f}")
print(f"macro_f1={test_macro_f1:.4f}")

avg_predicted_labels = test_preds.sum(axis=1).mean()
avg_true_labels = test_targets.sum(axis=1).mean()

print(f"\nAverage number of true labels per example: {avg_true_labels:.4f}")
print(f"Average number of predicted labels per example: {avg_predicted_labels:.4f}")

# Per theme F1 calculate
per_theme_rows = []
for idx, theme in enumerate(selected_themes):
    y_true = test_targets[:, idx]
    y_pred = test_preds[:, idx]

    theme_f1 = f1_score(y_true, y_pred, zero_division=0)
    support = int(y_true.sum())

    per_theme_rows.append({
        "theme": theme,
        "support": support,
        "f1": theme_f1
    })

per_theme_df = pd.DataFrame(per_theme_rows).sort_values("f1", ascending=False)

print("\nTop 15 best scoring themes:")
print(per_theme_df.head(15))

print("\nTop 15 worst scoring themes:")
print(per_theme_df.tail(15))

# Save results
per_theme_df.to_csv("outputs_multilabel/test_per_theme_metrics_baseline.csv", index=False)

summary = {
    "threshold": float(best_threshold),
    "test_loss": float(test_loss),
    "test_micro_f1": float(test_micro_f1),
    "test_macro_f1": float(test_macro_f1),
    "avg_true_labels_per_example": float(avg_true_labels),
    "avg_predicted_labels_per_example": float(avg_predicted_labels)
}

with open("outputs_multilabel/baseline_multilabel_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nFiles saved:")
print("- outputs_multilabel/test_per_theme_metrics_baseline.csv")
print("- outputs_multilabel/baseline_multilabel_summary.json")

### Building the weighted EfficientNet-B0 model for multilabel classification

In this step, we first calculate a separate positive class weight for each theme based on the training data. These weights are derived from the imbalance between positive and negative examples per label and are used to make the loss function more sensitive to rarer themes. This is especially relevant in multilabel classification, where some labels may appear much less frequently than others.

We then build a pretrained EfficientNet-B0 model and adapt it to the multilabel setting by replacing the final classifier layer with a new output layer of the correct size. As in the multiclass transfer learning setup, the feature extractor is frozen so that only the final classification head is trained in this first phase.

Next, we define a weighted multilabel loss function using the calculated class weights and set up the optimizer so that only trainable parameters are updated. This combines transfer learning with an explicit strategy to handle class imbalance in the multilabel target space.

Finally, we inspect the model structure, verify how many parameters are trainable, run a small forward-pass sanity check, and save the positive class weight table to disk. This confirms that the weighted EfficientNet model is correctly prepared before training begins.

In [ ]:
# calculating pos_weight for each theme based on training data
train_target_matrix = np.stack(train_df["target_vector"].values)  # shape: [n_samples, num_labels]

positive_counts = train_target_matrix.sum(axis=0)
negative_counts = len(train_target_matrix) - positive_counts

# Avoid deviding by zero
positive_counts = np.clip(positive_counts, a_min=1.0, a_max=None)

pos_weight = negative_counts / positive_counts
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(device)

pos_weight_df = pd.DataFrame({
    "theme": selected_themes,
    "positive_count": positive_counts.astype(int),
    "negative_count": negative_counts.astype(int),
    "pos_weight": pos_weight
}).sort_values("pos_weight", ascending=False)

print("Top 15 highest pos_weight themes:")
print(pos_weight_df.head(15))

print("\nTop 15 lowest pos_weight themes:")
print(pos_weight_df.tail(15))

# Build EfficientNet-B0 multilabel model
effnet_multilabel_model = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

# Freeze backbone
for param in effnet_multilabel_model.features.parameters():
    param.requires_grad = False

# Change classifier for multilabel output
in_features = effnet_multilabel_model.classifier[1].in_features
effnet_multilabel_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_labels)
)

effnet_multilabel_model = effnet_multilabel_model.to(device)

# Weighted multilabel loss + optimizer

effnet_multilabel_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

effnet_multilabel_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, effnet_multilabel_model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

print("\nEfficientNet multilabel model:")
print(effnet_multilabel_model)

# Paramter control
total_params = sum(p.numel() for p in effnet_multilabel_model.parameters())
trainable_params = sum(p.numel() for p in effnet_multilabel_model.parameters() if p.requires_grad)

print(f"\nTotal number of parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Sanity check on 1 batch

effnet_multilabel_model.eval()
with torch.no_grad():
    sample_logits = effnet_multilabel_model(images.to(device))

print("\nOutput shape:", sample_logits.shape)
print("Expected shape:", (images.shape[0], num_labels))

sample_probs = torch.sigmoid(sample_logits[:2]).cpu()
print("\nExample of sigmoid probabilities for the first 2 examples:")
print(sample_probs[:, :10])

# Save pos_weights
pos_weight_df.to_csv("outputs_multilabel/pos_weight_table.csv", index=False)

print("\nFile saved:")
print("- outputs_multilabel/pos_weight_table.csv")

### Training the weighted EfficientNet-B0 multilabel model

In this step, we train the weighted EfficientNet-B0 model on the multilabel task and evaluate it after each epoch on the validation set. The model is monitored using a fixed threshold, so that the multilabel predictions can be converted into binary outputs during training and validation.

As before, we store the full training history, save it to disk, and determine the best epoch based on validation macro-F1. This allows us to select the strongest model in a consistent way and to compare it fairly with the baseline multilabel CNN.

Finally, we visualise the learning curves for loss, micro-F1, and macro-F1 on the training and validation sets. These plots help us evaluate whether the pretrained weighted model is learning effectively and whether it improves upon the baseline approach.

In [ ]:
MONITOR_THRESHOLD = 0.08

# Train EfficientNet multilabel
effnet_multilabel_history = train_model_multilabel(
    model=effnet_multilabel_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=effnet_multilabel_criterion,
    optimizer=effnet_multilabel_optimizer,
    device=device,
    epochs=10,
    patience=3,
    threshold=MONITOR_THRESHOLD,
    model_path="outputs_multilabel/effnet_multilabel_best.pt"
)

print("\nTrainings history EfficientNet multilabel:")
print(effnet_multilabel_history)

# Save history
effnet_multilabel_history.to_csv(
    "outputs_multilabel/effnet_multilabel_history.csv",
    index=False
)
print("\nHistory saved as outputs_multilabel/effnet_multilabel_history.csv")

# Decide best epoch
best_idx = effnet_multilabel_history["val_macro_f1"].idxmax()
best_row = effnet_multilabel_history.loc[best_idx]

print("\nBest epoch on validation macro-F1:")
print(best_row)

# Plot 1: loss
plt.figure(figsize=(8, 5))
plt.plot(effnet_multilabel_history["epoch"], effnet_multilabel_history["train_loss"], label="Train loss")
plt.plot(effnet_multilabel_history["epoch"], effnet_multilabel_history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("EfficientNet Multilabel - Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: micro-F1
plt.figure(figsize=(8, 5))
plt.plot(effnet_multilabel_history["epoch"], effnet_multilabel_history["train_micro_f1"], label="Train micro-F1")
plt.plot(effnet_multilabel_history["epoch"], effnet_multilabel_history["val_micro_f1"], label="Validation micro-F1")
plt.xlabel("Epoch")
plt.ylabel("Micro-F1")
plt.title("EfficientNet Multilabel - Micro-F1")
plt.legend()
plt.tight_layout()
plt.show()

# Plot 3: macro-F1
plt.figure(figsize=(8, 5))
plt.plot(effnet_multilabel_history["epoch"], effnet_multilabel_history["train_macro_f1"], label="Train macro-F1")
plt.plot(effnet_multilabel_history["epoch"], effnet_multilabel_history["val_macro_f1"], label="Validation macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("EfficientNet Multilabel - Macro-F1")
plt.legend()
plt.tight_layout()
plt.show()

### Tuning the prediction threshold for EfficientNet-B0 in the multilabel setting

In this step, we reload the best EfficientNet-B0 multilabel model and collect its predicted probabilities on the validation set. As in the baseline multilabel setup, we do not immediately rely on a default threshold, but instead evaluate a range of possible thresholds to determine which decision rule works best for this model.

For each threshold, we calculate validation micro-F1, validation macro-F1, and the average number of predicted labels per example. This allows us to assess not only overall multilabel performance, but also how strongly the threshold affects the balance between missing labels and overpredicting labels.

We then identify the best threshold according to both macro-F1 and micro-F1, save the threshold tuning results, and visualise how the metrics evolve across different threshold values. This gives us a well-supported basis for selecting the final threshold before evaluating EfficientNet-B0 on the test set.

In [ ]:
# Reload best EfficientNet-model
best_effnet_multilabel_model = models.efficientnet_b0(
    weights=None
)

in_features = best_effnet_multilabel_model.classifier[1].in_features
best_effnet_multilabel_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_labels)
)

best_effnet_multilabel_model = best_effnet_multilabel_model.to(device)
best_effnet_multilabel_model.load_state_dict(
    torch.load("outputs_multilabel/effnet_multilabel_best.pt", map_location=device)
)

# Helper: collect probabilities and targets
@torch.no_grad()
def collect_probs_and_targets(model, loader, device):
    model.eval()
    all_probs = []
    all_targets = []

    for images, targets in loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_targets.append(targets.numpy())

    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)
    return all_probs, all_targets

val_probs, val_targets = collect_probs_and_targets(
    best_effnet_multilabel_model,
    val_loader,
    device
)

# Threshold grid
thresholds = [0.01, 0.02, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

rows = []
for threshold in thresholds:
    val_preds = (val_probs >= threshold).astype(np.float32)

    micro_f1 = f1_score(val_targets, val_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)
    avg_predicted_labels = val_preds.sum(axis=1).mean()

    rows.append({
        "threshold": threshold,
        "val_micro_f1": micro_f1,
        "val_macro_f1": macro_f1,
        "avg_predicted_labels_per_example": avg_predicted_labels
    })

effnet_threshold_results = pd.DataFrame(rows)

print("Threshold tuning results for EfficientNet multilabel:")
print(effnet_threshold_results)

# Best threshold based on macro-F1
best_idx_macro = effnet_threshold_results["val_macro_f1"].idxmax()
best_macro_row = effnet_threshold_results.loc[best_idx_macro]

# Best threshold based on micro-F1
best_idx_micro = effnet_threshold_results["val_micro_f1"].idxmax()
best_micro_row = effnet_threshold_results.loc[best_idx_micro]

print("\nBest threshold on validation macro-F1:")
print(best_macro_row)

print("\nBest threshold on validation micro-F1:")
print(best_micro_row)

# Opslaan
effnet_threshold_results.to_csv(
    "outputs_multilabel/effnet_threshold_tuning_results.csv",
    index=False
)

# Plot micro-F1
plt.figure(figsize=(8, 5))
plt.plot(effnet_threshold_results["threshold"], effnet_threshold_results["val_micro_f1"], marker="o")
plt.xlabel("Threshold")
plt.ylabel("Validation micro-F1")
plt.title("EfficientNet multilabel - threshold tuning (micro-F1)")
plt.tight_layout()
plt.show()

# Plot macro-F1
plt.figure(figsize=(8, 5))
plt.plot(effnet_threshold_results["threshold"], effnet_threshold_results["val_macro_f1"], marker="o")
plt.xlabel("Threshold")
plt.ylabel("Validation macro-F1")
plt.title("EfficientNet multilabel - threshold tuning (macro-F1)")
plt.tight_layout()
plt.show()

# Plot average number of predicted labels
plt.figure(figsize=(8, 5))
plt.plot(
    effnet_threshold_results["threshold"],
    effnet_threshold_results["avg_predicted_labels_per_example"],
    marker="o"
)
plt.xlabel("Threshold")
plt.ylabel("Average number of predicted labels")
plt.title("EfficientNet multilabel - predicted labels per example")
plt.tight_layout()
plt.show()

print("\nFile saved:")
print("- outputs_multilabel/effnet_threshold_tuning_results.csv")

### Evaluating the final EfficientNet-B0 multilabel model on the test set

In this step, we evaluate the best EfficientNet-B0 multilabel model on the hold-out test set using the selected decision threshold. This gives us the final test performance of the pretrained multilabel model in terms of loss, micro-F1, and macro-F1.

We also compare the average number of true labels and predicted labels per example. This is especially relevant in multilabel classification, because it helps us understand whether the model predicts a realistic number of labels per minifig or whether it tends to overpredict or underpredict the active themes.

Next, we calculate per-theme F1-scores and sort the labels from best to worst. This provides a detailed view of how the model performs on individual themes and makes it easier to identify which labels benefit most from the pretrained architecture and which remain difficult. Finally, we save the per-theme results, the top and bottom theme rankings, and a compact summary of the main test metrics to disk.

In [ ]:
efficientnet_test_threshold = 0.50

# Rebuild and load best EfficientNet-model
best_effnet_multilabel_model = models.efficientnet_b0(weights=None)

in_features = best_effnet_multilabel_model.classifier[1].in_features
best_effnet_multilabel_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_labels)
)

best_effnet_multilabel_model = best_effnet_multilabel_model.to(device)
best_effnet_multilabel_model.load_state_dict(
    torch.load("outputs_multilabel/effnet_multilabel_best.pt", map_location=device)
)

# Execute test evaluation
test_loss, test_micro_f1, test_macro_f1, test_targets, test_preds, test_probs = evaluate_multilabel(
    best_effnet_multilabel_model,
    test_loader,
    effnet_multilabel_criterion,
    device,
    threshold=efficientnet_test_threshold
)

avg_true_labels = test_targets.sum(axis=1).mean()
avg_predicted_labels = test_preds.sum(axis=1).mean()

print("EfficientNet multilabel - Test results:")
print(f"Threshold: {efficientnet_test_threshold:.2f}")
print(f"loss={test_loss:.4f}")
print(f"micro_f1={test_micro_f1:.4f}")
print(f"macro_f1={test_macro_f1:.4f}")

print(f"\nAverage amount of true labels per example: {avg_true_labels:.4f}")
print(f"Average amount of predicted labels per example: {avg_predicted_labels:.4f}")

# Calculate per-theme metrics

per_theme_rows = []

for idx, theme in enumerate(selected_themes):
    y_true = test_targets[:, idx]
    y_pred = test_preds[:, idx]

    theme_f1 = f1_score(y_true, y_pred, zero_division=0)
    support = int(y_true.sum())

    per_theme_rows.append({
        "theme": theme,
        "support": support,
        "f1": theme_f1
    })

effnet_test_per_theme_df = pd.DataFrame(per_theme_rows)

# For an overview of results, we sort the per-theme table in 2 ways:
# - best scoring themes: highest F1 first
# - worst scoring themes: lowest F1 first
# At equal F1 we sort on support
effnet_test_per_theme_sorted_desc = effnet_test_per_theme_df.sort_values(
    by=["f1", "support", "theme"],
    ascending=[False, False, True]
).reset_index(drop=True)

effnet_test_per_theme_sorted_asc = effnet_test_per_theme_df.sort_values(
    by=["f1", "support", "theme"],
    ascending=[True, True, True]
).reset_index(drop=True)

top_15_best_themes = effnet_test_per_theme_sorted_desc.head(15)
top_15_worst_themes = effnet_test_per_theme_sorted_asc.head(15)

print("\nTop 15 best scoring themes:")
print(top_15_best_themes.to_string(index=False))

print("\nTop 15 worst scoring themes:")
print(top_15_worst_themes.to_string(index=False))

print("\nFull per-theme table:")
print(
    effnet_test_per_theme_sorted_desc[["theme", "support", "f1"]]
    .to_string(index=False)
)

# Save results
effnet_test_per_theme_df.to_csv(
    "outputs_multilabel/effnet_test_per_theme_metrics.csv",
    index=False
)

top_15_best_themes.to_csv(
    "outputs_multilabel/effnet_test_top15_best_themes.csv",
    index=False
)

top_15_worst_themes.to_csv(
    "outputs_multilabel/effnet_test_top15_worst_themes.csv",
    index=False
)

effnet_test_summary = {
    "threshold": float(efficientnet_test_threshold),
    "test_loss": float(test_loss),
    "test_micro_f1": float(test_micro_f1),
    "test_macro_f1": float(test_macro_f1),
    "avg_true_labels_per_example": float(avg_true_labels),
    "avg_predicted_labels_per_example": float(avg_predicted_labels)
}

with open("outputs_multilabel/effnet_multilabel_test_summary.json", "w", encoding="utf-8") as f:
    json.dump(effnet_test_summary, f, indent=2)

### Comparing the baseline multilabel CNN with EfficientNet-B0

In this final comparison step, we bring together the results of the baseline multilabel CNN and the pretrained EfficientNet-B0 model. We load the saved summaries and per-theme evaluation tables so that both models can be compared in a consistent way on the hold-out test set.

The comparison is performed at two levels. First, we compare the global test metrics, such as loss, micro-F1, macro-F1, and the average number of predicted labels per example. Second, we compare both models at the theme level by analysing how much the per-theme F1-score improves or decreases when moving from the baseline model to EfficientNet-B0.

This gives a complete overview of whether the pretrained model provides a meaningful improvement, not only in overall performance but also across the individual multilabel themes. Finally, the results are visualised and saved to disk so they can be reused in the final discussion.

In [ ]:
# Load summaries and per-theme results for comparison
with open("outputs_multilabel/baseline_multilabel_summary.json", "r", encoding="utf-8") as f:
    baseline_summary = json.load(f)

with open("outputs_multilabel/effnet_multilabel_test_summary.json", "r", encoding="utf-8") as f:
    effnet_summary = json.load(f)

baseline_per_theme = pd.read_csv("outputs_multilabel/test_per_theme_metrics_baseline.csv")
effnet_per_theme = pd.read_csv("outputs_multilabel/effnet_test_per_theme_metrics.csv")

# Global comparison on test set
comparison_df = pd.DataFrame([
    {
        "model": "Baseline CNN",
        "threshold": baseline_summary["threshold"],
        "test_loss": baseline_summary["test_loss"],
        "test_micro_f1": baseline_summary["test_micro_f1"],
        "test_macro_f1": baseline_summary["test_macro_f1"],
        "avg_true_labels_per_example": baseline_summary["avg_true_labels_per_example"],
        "avg_predicted_labels_per_example": baseline_summary["avg_predicted_labels_per_example"],
    },
    {
        "model": "EfficientNet-B0",
        "threshold": effnet_summary["threshold"],
        "test_loss": effnet_summary["test_loss"],
        "test_micro_f1": effnet_summary["test_micro_f1"],
        "test_macro_f1": effnet_summary["test_macro_f1"],
        "avg_true_labels_per_example": effnet_summary["avg_true_labels_per_example"],
        "avg_predicted_labels_per_example": effnet_summary["avg_predicted_labels_per_example"],
    }
])

print("Global comparison on test set:")
print(comparison_df.to_string(index=False))

# Extra difference row
delta_row = pd.DataFrame([{
    "model": "Difference (EfficientNet - Baseline)",
    "threshold": effnet_summary["threshold"] - baseline_summary["threshold"],
    "test_loss": effnet_summary["test_loss"] - baseline_summary["test_loss"],
    "test_micro_f1": effnet_summary["test_micro_f1"] - baseline_summary["test_micro_f1"],
    "test_macro_f1": effnet_summary["test_macro_f1"] - baseline_summary["test_macro_f1"],
    "avg_true_labels_per_example": effnet_summary["avg_true_labels_per_example"] - baseline_summary["avg_true_labels_per_example"],
    "avg_predicted_labels_per_example": effnet_summary["avg_predicted_labels_per_example"] - baseline_summary["avg_predicted_labels_per_example"],
}])

print("\nDifference (EfficientNet - Baseline):")
print(delta_row.to_string(index=False))

# Per-theme comparison
baseline_per_theme = baseline_per_theme.rename(columns={
    "f1": "baseline_f1",
    "support": "support_baseline"
})

effnet_per_theme = effnet_per_theme.rename(columns={
    "f1": "effnet_f1",
    "support": "support_effnet"
})

per_theme_comparison = baseline_per_theme.merge(
    effnet_per_theme,
    on="theme",
    how="inner"
)

# Support should be identical; we keep one support-column
per_theme_comparison["support"] = per_theme_comparison["support_effnet"]

per_theme_comparison = per_theme_comparison[[
    "theme",
    "support",
    "baseline_f1",
    "effnet_f1"
]].copy()

per_theme_comparison["f1_improvement"] = (
    per_theme_comparison["effnet_f1"] - per_theme_comparison["baseline_f1"]
)

per_theme_comparison = per_theme_comparison.sort_values(
    by="f1_improvement",
    ascending=False
).reset_index(drop=True)

print("\nTop 15 highest improving themes:")
print(per_theme_comparison.head(15).to_string(index=False))

print("\nTop 15 lowest improving themes:")
print(per_theme_comparison.tail(15).to_string(index=False))

# --------------------------------------------------
# 4. Enkele samenvattende tellingen
# --------------------------------------------------
n_better = (per_theme_comparison["f1_improvement"] > 0).sum()
n_equal = (per_theme_comparison["f1_improvement"] == 0).sum()
n_worse = (per_theme_comparison["f1_improvement"] < 0).sum()

print("\nNumber of themes on which EfficientNet scores better / equal / worse:")
print(f"Beter : {n_better}")
print(f"Gelijk: {n_equal}")
print(f"Slechter: {n_worse}")

# Sort on EfficientNet F1, then support, then theme name
best_effnet_themes = per_theme_comparison.sort_values(
    by=["effnet_f1", "support", "theme"],
    ascending=[False, False, True]
).head(15)

worst_effnet_themes = per_theme_comparison.sort_values(
    by=["effnet_f1", "support", "theme"],
    ascending=[True, True, True]
).head(15)

print("\nTop 15 themes according to EfficientNet F1 (with baseline reference):")
print(best_effnet_themes.to_string(index=False))

print("\nWorst 15 themes according to EfficientNet F1 (with baseline reference):")
print(worst_effnet_themes.to_string(index=False))

# Visualistions
# Plot 1: global comparison on micro/macro-F1
plt.figure(figsize=(8, 5))
plot_df = comparison_df.set_index("model")[["test_micro_f1", "test_macro_f1"]]
plot_df.plot(kind="bar")
plt.ylabel("F1-score")
plt.title("Baseline vs EfficientNet op de testset")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Plot 2: average number of predicted labels
plt.figure(figsize=(8, 5))
plt.bar(comparison_df["model"], comparison_df["avg_predicted_labels_per_example"])
plt.ylabel("Average number of predicted labels per example")
plt.title("Predicted number of labels per model")
plt.tight_layout()
plt.show()

# Plot 3: top 15 highest improvements
top15_improvement = per_theme_comparison.head(15).sort_values("f1_improvement", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top15_improvement["theme"], top15_improvement["f1_improvement"])
plt.xlabel("F1-improvement (EfficientNet - Baseline)")
plt.title("Top 15 highest improvements per theme")
plt.tight_layout()
plt.show()

# Save results
comparison_df.to_csv(
    "outputs_multilabel/model_comparison_baseline_vs_effnet.csv",
    index=False
)

delta_row.to_csv(
    "outputs_multilabel/model_comparison_deltas.csv",
    index=False
)

per_theme_comparison.to_csv(
    "outputs_multilabel/per_theme_comparison_baseline_vs_effnet.csv",
    index=False
)

best_effnet_themes.to_csv(
    "outputs_multilabel/best_effnet_themes_with_baseline_reference.csv",
    index=False
)

worst_effnet_themes.to_csv(
    "outputs_multilabel/worst_effnet_themes_with_baseline_reference.csv",
    index=False
)

comparison_summary = {
    "baseline_threshold": float(baseline_summary["threshold"]),
    "effnet_threshold": float(effnet_summary["threshold"]),
    "baseline_test_micro_f1": float(baseline_summary["test_micro_f1"]),
    "effnet_test_micro_f1": float(effnet_summary["test_micro_f1"]),
    "baseline_test_macro_f1": float(baseline_summary["test_macro_f1"]),
    "effnet_test_macro_f1": float(effnet_summary["test_macro_f1"]),
    "micro_f1_improvement": float(effnet_summary["test_micro_f1"] - baseline_summary["test_micro_f1"]),
    "macro_f1_improvement": float(effnet_summary["test_macro_f1"] - baseline_summary["test_macro_f1"]),
    "baseline_avg_predicted_labels": float(baseline_summary["avg_predicted_labels_per_example"]),
    "effnet_avg_predicted_labels": float(effnet_summary["avg_predicted_labels_per_example"]),
    "predicted_labels_difference": float(
        effnet_summary["avg_predicted_labels_per_example"] - baseline_summary["avg_predicted_labels_per_example"]
    ),
    "n_themes_effnet_better": int(n_better),
    "n_themes_equal": int(n_equal),
    "n_themes_effnet_worse": int(n_worse),
}

with open("outputs_multilabel/model_comparison_summary.json", "w", encoding="utf-8") as f:
    json.dump(comparison_summary, f, indent=2)

print("\nFiles saved:")
print("- outputs_multilabel/model_comparison_baseline_vs_effnet.csv")
print("- outputs_multilabel/model_comparison_deltas.csv")
print("- outputs_multilabel/per_theme_comparison_baseline_vs_effnet.csv")
print("- outputs_multilabel/best_effnet_themes_with_baseline_reference.csv")
print("- outputs_multilabel/worst_effnet_themes_with_baseline_reference.csv")
print("- outputs_multilabel/model_comparison_summary.json")

## Summary of part 2 of this task

In this part of the notebook, we reformulated the Lego Minifigs problem as a multilabel classification task, where each image can be associated with multiple themes instead of exactly one category. We first prepared the metadata, cleaned the dataset, explored the theme distribution, and selected a subset of themes with sufficient support to define a stable multilabel target space.

Next, we encoded the selected themes as multi-hot target vectors and created train, validation, and test splits. After building the multilabel dataset and dataloaders, we trained a baseline CNN and evaluated it using micro-F1 and macro-F1. Because multilabel predictions depend strongly on the decision threshold, we also tuned the prediction threshold on the validation set before performing the final test evaluation.

We then extended the approach with a pretrained EfficientNet-B0 model combined with a weighted multilabel loss to handle label imbalance more explicitly. After training and threshold tuning, we evaluated this pretrained model on the test set and compared it directly with the baseline model, both globally and at the level of individual themes.

Overall, this multilabel pipeline provides a structured and well-documented approach to predicting multiple themes per minifig image, while also allowing a fair comparison between a simple baseline model and a stronger pretrained alternative.

## Final conclusion

This notebook addressed Task 2 in two different settings: a multiclass classification setup, where each minifig is assigned to one category, and a multilabel classification setup, where each minifig can be linked to multiple themes. In both parts, we followed the same general workflow: data preparation, target construction, train-validation-test splitting, model building, training, evaluation, and model comparison.

The multiclass part showed that image-based category prediction is feasible, but also that performance depends strongly on the strength of the model architecture. The baseline CNN provided a useful reference, while the pretrained EfficientNet-B0 model offered a stronger solution for handling the complexity of the category structure.

The multilabel part extended this analysis by allowing multiple active labels per image. This required a different target representation, a different loss function, and explicit threshold tuning. Here as well, the pretrained EfficientNet-B0 model provided the more advanced solution, especially because it combined transfer learning with a weighted loss to better deal with label imbalance.

Taken together, the results show that pretrained convolutional models are a more effective choice than simple CNN baselines for both the multiclass and multilabel versions of the Lego Minifigs problem. The notebook therefore demonstrates not only how the pipeline was implemented, but also why more modern pretrained architectures are better suited for these image classification tasks.